# RDD Partitions, Partitioners, and Parallelism Inspection

**Module**: 02_rdd  
**Notebook**: 02_rdd_partitions.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How RDDs are sliced into partitions, inspecting partition distribution with glom(), and controlling parallelism.
- **Why does this topic exist?**: Partitions are the atomic unit of parallelism in Spark. Misconfigured partitions lead to either idle cores or memory thrashing.
- **What problem does it solve?**: Under-partitioning wastes cluster resources; over-partitioning causes scheduling bottleneck.
- **Where is it used in real-world Data Engineering?**: Optimizing batch ETL runtimes and tuning data ingestion from external file systems.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | RDD transformations |
| **Required depth** | Working knowledge: can inspect and control partition distribution |
| **Data Engineer** | Inspect partitions with `glom()`, choose `repartition` vs `coalesce`, and explain parallelism. |
| **Senior Data Engineer** | Diagnose skew and small-task overhead from partition layout. |
| **Spark specialist** | Reason about partitioners, shuffle boundaries, and task granularity. |
| **Next topic** | `02_rdd/04_rdd_dependencies_and_lineage.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Partition count is the main lever for parallelism, but more partitions are not always better. Too few
partitions leave cores idle; too many create scheduling overhead and tiny tasks; uneven partitions create
stragglers. Partitioning is therefore a data-layout decision, not a cosmetic setting.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Partition** | A chunk of an RDD | Unit of task input and parallelism |
| **Partitioner** | How keys map to partitions | HashPartitioner, RangePartitioner, custom |
| **Parallelism** | How many tasks can run at once | Bounded by cores and partition count |
| **`glom()`** | Inspect partition contents | Returns an RDD of lists, one per partition |
| **`repartition`** | Change partition count with a shuffle | Can increase or decrease; full exchange |
| **`coalesce`** | Reduce partitions without a full shuffle | Merges local partitions; can be uneven |

### How it works, step by step

1. The source or `parallelize` call sets the initial partition count.
2. A transformation inherits the parent partitioning unless it is wide.
3. A wide transformation uses a partitioner to route keys.
4. The scheduler creates one task per partition per stage.
5. Idle cores mean too few partitions or a resource bottleneck.
6. Tiny tasks mean too many partitions or small files.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Partition Mechanics

```
RDD (Distributed Collection)
+---------------------------------------------------------------+
|  Partition 0  |  Partition 1  |  Partition 2  |  Partition 3  |
+---------------+---------------+---------------+---------------+
        |               |               |               |
        v               v               v               v
    Task 0          Task 1          Task 2          Task 3
 (Core 1, W1)    (Core 2, W1)    (Core 1, W2)    (Core 2, W2)
```
- Each partition is assigned to exactly **one Task** running on **one CPU Core**.
- Default partition count when reading files is determined by total input splits (default 128 MB blocks).

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("RDD_Partitions_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Inspecting Partitions with `getNumPartitions()` and `glom()`

In [ ]:
sc = spark.sparkContext
numbers = list(range(1, 25))

# Create RDD with 4 explicit partitions
rdd_4 = sc.parallelize(numbers, 4)
print(f"Partition count: {rdd_4.getNumPartitions()}")

# glom() gathers all elements of each partition into a Python list
partition_contents = rdd_4.glom().collect()
for idx, p in enumerate(partition_contents):
    print(f"  Partition {idx} (size {len(p)}): {p}")

## 4. `mapPartitions` and `mapPartitionsWithIndex`

`mapPartitions` processes an entire partition iterator at once, enabling connection reuse and batch operations.

In [ ]:
def compute_partition_stats(index, iterator):
    items = list(iterator)
    if items:
        yield {
            "partition": index,
            "count": len(items),
            "min": min(items),
            "max": max(items),
            "sum": sum(items)
        }

stats = rdd_4.mapPartitionsWithIndex(compute_partition_stats).collect()
print("Partition Statistics:")
for s in stats:
    print(" ", s)

---

## Experiment: Execution Latency vs Partition Count

- **1. What are we testing?**: Benchmark how execution duration varies when running on 1 partition vs 4 partitions on a 4-core cluster.
- **2. What do we expect?**: 1 partition utilizes only 1 core (sequential); 4 partitions utilize all 4 cores concurrently (~4x speedup).

In [ ]:
import math

def heavy_computation(x):
    # CPU-bound calculation
    return sum(math.sqrt(i) for i in range(1, 10000))

data = list(range(200))

# 1 Partition (Sequential)
t0 = time.time()
sc.parallelize(data, 1).map(heavy_computation).count()
t_1 = time.time() - t0

# 4 Partitions (Distributed across all 4 worker cores)
t0 = time.time()
sc.parallelize(data, 4).map(heavy_computation).count()
t_4 = time.time() - t0

print(f"1 Partition duration:  {t_1:.3f}s (1 core active, 3 idle)")
print(f"4 Partitions duration: {t_4:.3f}s (all 4 cores active)")
print(f"Speedup factor:        {t_1 / t_4:.2f}x")

**3. What happened?**: 4 partitions ran nearly 3-4x faster than 1 partition.  
**4. Why did it happen?**: Because work was distributed in parallel across all 4 available executor cores in our 2-worker cluster.  
**5. Practical Engineering Lesson**: Always ensure partition count is at least equal to or a multiple of total executor cores.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Ingesting many small files

- **Industry**: Retail / data lake
- **Scenario**: A daily ingestion job reads 20,000 small JSON files and creates one partition per file.
- **Problem**: The job spends more time scheduling tasks than processing data.
- **Why this concept is relevant**: Repartitioning after ingestion creates fewer, larger partitions and reduces task overhead.
- **Implementation idea**: Read the files, `repartition` to a target based on data size and cores, then write Parquet.
- **Expected result**: Fewer tasks and larger output files.
- **Alternative**: A file compaction job or a table format with automatic compaction.
- **Trade-offs**: Spark repartition is simple; table formats handle compaction more continuously.

### Use Case 2: Custom partitioning for a hot key

- **Industry**: Advertising / clickstream
- **Scenario**: A campaign ID is extremely hot, and hash partitioning sends most events to one reducer.
- **Problem**: One task becomes a straggler while the rest of the cluster is idle.
- **Why this concept is relevant**: A custom partitioner or salting can spread the hot key across partitions.
- **Implementation idea**: Inspect key frequencies, salt the hot key, aggregate partially, then combine.
- **Expected result**: A balanced stage with a second aggregation to restore the final result.
- **Alternative**: AQE skew join handling or a dedicated streaming aggregation.
- **Trade-offs**: Manual salting is transparent but adds complexity; AQE is automatic but only applies to supported joins.

---

## Business Value

- Better cluster utilization and predictable runtimes.
- Lower scheduling overhead for small-file workloads.
- Fewer stragglers and OOMs from skew.
- More stable SLAs for recurring pipelines.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Aim for partitions large enough to amortize task overhead but small enough to fit memory.
- Use `coalesce` to reduce partitions without a shuffle; use `repartition` when you need a full redistribution.
- Inspect partition sizes with `glom()` or the Spark UI task input size.
- Avoid one partition for large data and thousands of tiny partitions.
- Check `spark.default.parallelism` and `spark.sql.shuffle.partitions` separately.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Partition count vs runtime

**Steps**

1. Create an RDD of 1,000,000 integers with 1, 4, 8, and 16 partitions.
2. Time a simple `map` + `count` for each.
3. Inspect task duration and scheduling overhead in the UI.

**What to observe**: Too few partitions underutilize cores; too many add overhead.

**Healthy result**: A partition count near a small multiple of available cores performs well for CPU-bound work.

**Problematic result**: One partition serializes the work; thousands of partitions create tiny tasks.

**Engineering takeaway**: Measure the trade-off on your own cluster.

---

## Trade-offs and Alternatives

### Advantages

- Direct control over parallelism
- Can fix skew and small-task overhead
- Easy to inspect

### Limitations

- Repartition is expensive
- Coalesce can be uneven
- Optimal size depends on workload

### When to use it

- Before a large aggregation
- After reading many small files
- When tuning task granularity

### When not to use it

- Repartitioning repeatedly
- Coalescing to one partition for large outputs
- Guessing without metrics

### Alternatives inside Spark

- `repartition`
- `coalesce`
- AQE coalescing

### Alternatives outside Spark

- File compaction
- Table format compaction
- Source-side partitioning

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Partitions of Retail Orders RDD
**Problem Statement**: Load /opt/spark-data/retail/orders.csv via textFile and print its default partition count.  
**Input / Context**: sc.textFile('/opt/spark-data/retail/orders.csv')

In [ ]:
# Exercise 1: Orders partition count

### Exercise 2 (Intermediate): Find Largest and Smallest Partition via glom()
**Problem Statement**: Use glom() and map() to calculate the minimum and maximum partition sizes for orders_rdd.  
**Input / Context**: orders_rdd

In [ ]:
# Exercise 2: Min and max partition sizes

### Exercise 3 (Advanced): Custom Hash Partitioning with partitionBy()
**Problem Statement**: Create a PairRDD of (customer_id, order_id) and partition it into 4 partitions using HashPartitioner.  
**Input / Context**: orders_rdd mapped to (customer_id, order_id)

In [ ]:
# Exercise 3: partitionBy

---

## Production Debugging Exercise

### The single-partition bottleneck

**Scenario**: A job uses `coalesce(1)` before a heavy aggregation and one task takes 40 minutes.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the number of tasks in the stage.
2. Look at the largest partition size.
3. Remove the coalesce or repartition before the aggregation.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Partitions of Retail Orders RDD):
```python
orders_rdd = sc.textFile("/opt/spark-data/retail/orders.csv")
print("Default partitions for orders.csv:", orders_rdd.getNumPartitions())
```

#### Solution 2 (Find Largest and Smallest Partition via glom()):
```python
sizes = orders_rdd.glom().map(len).collect()
print("Partition sizes:", sizes)
print(f"Min partition size: {min(sizes)}, Max partition size: {max(sizes)}")
```

#### Solution 3 (Custom Hash Partitioning with partitionBy()):
```python
header = orders_rdd.first()
kv_rdd = orders_rdd.filter(lambda r: r != header).map(lambda r: r.split(",")).map(lambda c: (int(c[1]), int(c[0])))
partitioned_rdd = kv_rdd.partitionBy(4)
print("New partition count:", partitioned_rdd.getNumPartitions())
print("Distribution per partition:", partitioned_rdd.glom().map(len).collect())
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Partitions = Parallelism**: 1 task processes 1 partition. Max concurrency = total available executor cores.
- **glom() Utility**: Useful for diagnostics; gathers partition items into lists.
- **mapPartitions Efficiency**: Enables batch operations and connection reuse per partition.

### Essential Technical Terms
- `Partition`: Atomic split of distributed data.
- `glom()`: Collects elements of each partition into an array.
- `mapPartitions`: Operates on an entire partition via an iterator.

### Common Engineering Mistakes
- **Single Partition Data**: Processing large data in 1 partition serializes execution and causes OOM.
- **Calling glom().collect() on Huge Partitions**: Brings entire partition lists into Driver memory, causing OOM.

### Interview Questions on This Topic
- **Q: What is the relationship between partitions and tasks?**
  - *A*: There is a strict 1:1 mapping: each stage creates exactly one task for every partition in its input RDD.
- **Q: Why use mapPartitions instead of map?**
  - *A*: To avoid per-row initialization overhead (e.g. database connections, regex compilation, heavy parsers).




### More Interview Questions

- **Q: What is the difference between `repartition` and `coalesce`?**
  - *A*: `repartition` performs a full shuffle and can increase or decrease partitions; `coalesce` merges existing partitions locally and is cheaper but can be uneven.
- **Q: How does partition count affect tasks?**
  - *A*: Each stage creates one task per partition, so partition count controls task parallelism and granularity.

### What to Study Next
Open 02_rdd/04_rdd_dependencies_and_lineage.ipynb to explore RDD fault tolerance and stage boundaries.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')